In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from littleballoffur import ForestFireSampler
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
import torch
import pandas as pd
import itertools 
import os

In [2]:
def plot(G,attribute):
    color_class_map = {0: 'blue', 1: 'red', 2: 'darkgreen', 3: 'orange'}
    nx.draw(G, 
            with_labels=False, 

            node_color=[color_class_map[node[1][attribute]] 
                        for node in G.nodes(data=True)], 
            
           font_color='white',node_size=100)
    print('node count',G.number_of_nodes())
    print('edge count',G.number_of_edges())
    plt.show()

In [3]:
# Compute min, max, mean degree
def statistics_degrees(G):

    degrees = [G.degree(n) for n in G.nodes()]
    return np.max(degrees), np.min(degrees), np.mean(degrees)

In [4]:
import seaborn as sns

def plot_degree_dist(G):
    
    degrees = [G.degree(n) for n in G.nodes()]
    print(degrees)
    #plt.hist(degrees)
    sns.histplot(data=degrees)
    plt.xlabel('Degree')
    plt.ylabel('Frequency')
    plt.show()

## Original Graphs

In [5]:
file_path = '../../real_pubmed/15.pt'

In [6]:
adjs, node_types = torch.load(file_path)

In [7]:
len(adjs)

200

In [8]:
orig_max_node_degree_list = []
orig_min_node_degree_list = []
orig_mean_node_degree_list = []
orig_clus_coeff_list = []
for i, adj in enumerate(adjs[:115]):
    #print(adj)
    #print(node_types[i])
    #print(list(set(node_types[i])))
    types= list(node_types[i])
    if set([0,1,2,3]).issubset(types):
        #print(True)
        G = nx.from_numpy_array(adj.numpy())
        nx.set_node_attributes(G, dict(zip(G.nodes(), node_types[i])), 'node_type')
        #plot(G,'node_type')
        clus_coeff = nx.average_clustering(G)
        #avg_path_length = nx.average_shortest_path_length(G_syn)
        max, min, mean = statistics_degrees(G)
        
        orig_max_node_degree_list.append(max)
        orig_min_node_degree_list.append(min)
        orig_mean_node_degree_list.append(mean)
        orig_clus_coeff_list.append(clus_coeff)

In [9]:
len(orig_max_node_degree_list)

50

In [10]:
print('Average max node degree', np.mean(orig_max_node_degree_list))
print('Average min node degree', np.mean(orig_min_node_degree_list))
print('Average mean node degree', np.mean(orig_mean_node_degree_list))
print('Average clustering coefficient', np.mean(orig_clus_coeff_list))

Average max node degree 9.76
Average min node degree 1.02
Average mean node degree 3.445333333333333
Average clustering coefficient 0.2794463758463758


## Baseline VAE

### PubMed

In [11]:
vae_max_node_degree_list = []
vae_min_node_degree_list = []
vae_mean_node_degree_list = []
vae_clus_coeff_list = []

In [12]:
rootdir = 'pubmed_vae_15'
dir_list = []
for subdir, dirs,files in os.walk(rootdir):

    if files:
        filepath = os.path.join(subdir, files[0]) 
        #print(filepath)
        G_syn= nx.read_gexf(filepath)
        clus_coeff = nx.average_clustering(G_syn)
        #avg_path_length = nx.average_shortest_path_length(G_syn)
        max, min, mean = statistics_degrees(G_syn)
        
        vae_max_node_degree_list.append(max)
        vae_min_node_degree_list.append(min)
        vae_mean_node_degree_list.append(mean)
        vae_clus_coeff_list.append(clus_coeff)


In [13]:
len(vae_max_node_degree_list[:50])

50

In [14]:
print('Average max node degree', np.mean(vae_max_node_degree_list[:50]))

Average max node degree 7.22


In [15]:
#vae_min_node_degree_list

In [16]:
print('Average min node degree', np.mean(vae_min_node_degree_list[:50]))

Average min node degree 1.38


In [17]:
#vae_mean_node_degree_list

In [18]:
print('Average mean node degree', np.mean(vae_mean_node_degree_list[:50]))

Average mean node degree 4.245333333333333


In [19]:
#vae_clus_coeff_list

In [20]:
print('Average clustering coefficient', np.mean(vae_clus_coeff_list[:50]))

Average clustering coefficient 0.29706031746031747


## Diffusion

### PubMed

In [21]:
path = 'pubmed_syn_small_15/' 
files = os.listdir(path)

In [22]:
diff_max_node_degree_list = []
diff_min_node_degree_list = []
diff_mean_node_degree_list = []
diff_clus_coeff_list = []

In [23]:
for index, file in enumerate(files):
    
    if file.endswith('.gexf'):

        filepath = os.path.join(path, file)
        #print(filepath)
        G_syn= nx.read_gexf(filepath)
        clus_coeff = nx.average_clustering(G_syn)
        #avg_path_length = nx.average_shortest_path_length(G_syn)
        max, min, mean = statistics_degrees(G_syn)
        
        diff_max_node_degree_list.append(max)
        diff_min_node_degree_list.append(min)
        diff_mean_node_degree_list.append(mean)
        diff_clus_coeff_list.append(clus_coeff)

In [24]:
len(diff_max_node_degree_list)

50

In [25]:
#diff_max_node_degree_list

In [26]:
print('Average max node degree', np.mean(diff_max_node_degree_list))

Average max node degree 9.22


In [27]:
#diff_min_node_degree_list

In [28]:
print('Average min node degree', np.mean(diff_min_node_degree_list))

Average min node degree 1.0


In [29]:
#diff_mean_node_degree_list

In [30]:
print('Average mean node degree', np.mean(diff_mean_node_degree_list))

Average mean node degree 2.733333333333333


In [31]:
#diff_clus_coeff_list

In [32]:
print('Average clustering coefficient', np.mean(diff_clus_coeff_list))

Average clustering coefficient 0.2387753653753654
